In [ ]:
# --------- Importing libraries ---------
import pandas as pd
import json

In [ ]:
# -------- Defining file path & Target ----------
file_path = '../data/yelp_academic_dataset_business.json'
target_state = "LA" # Louisiana (New Orleans Metropolitan Area)

restaurants_data = []

print(f"Extrating restaurants for state: {target_state}...")

Extrating restaurants for state: LA...


In [ ]:
# --------- Raw Data Inspection ----------

print("Reading the first line of the raw file to inspect its structure...\n")

with open(file_path, 'r', encoding='utf-8') as file:
    # Read only the first line to prevent loading the entire file into memory
    first_line = file.readline()
    sample_business = json.loads(first_line)

# Print the discovered keys 
print("-> DISCOVERED VARIABLES (JSON KEYS):")
print(list(sample_business.keys()))

# Display the first record cleanly formatted
print("\n-> FIRST RECORD SAMPLE:")
print(json.dumps(sample_business, indent=4))

Reading the first line of the raw file to inspect its structure...

-> DISCOVERED VARIABLES (JSON KEYS):
['business_id', 'name', 'address', 'city', 'state', 'postal_code', 'latitude', 'longitude', 'stars', 'review_count', 'is_open', 'attributes', 'categories', 'hours']

-> FIRST RECORD SAMPLE:
{
    "business_id": "Pns2l4eNsfO8kk83dixA6A",
    "name": "Abby Rappoport, LAC, CMQ",
    "address": "1616 Chapala St, Ste 2",
    "city": "Santa Barbara",
    "state": "CA",
    "postal_code": "93101",
    "latitude": 34.4266787,
    "longitude": -119.7111968,
    "stars": 5.0,
    "review_count": 7,
    "is_open": 0,
    "attributes": {
        "ByAppointmentOnly": "True"
    },
    "categories": "Doctors, Traditional Chinese Medicine, Naturopathic/Holistic, Acupuncture, Health & Medical, Nutritionists",
    "hours": null
}


In [ ]:
# -------- Streaming & Filtering the JSON ---------

with open(file_path, 'r', encoding='utf-8') as file:
    for line in file:
        business = json.loads(line)

        # Extrating the variables
        state = business.get('state')
        categories = business.get('categories')

        # Filtering: Must be in 'LA' and must have 'Restaurants' or 'Food'
        if categories and state == target_state:
            if 'Restaurant' in categories and 'Food' in categories:

                # Flattening the 'attributes' nested dictionary
                # This turns a tree structure into flat columns (e.g., attr_WiFi)
                attributes = business.pop('attributes', {})
                if attributes:
                    for key, value in attributes.items():
                        business[f'attr_{key}'] = value
                restaurants_data.append(business)

In [ ]:
# ------- Creating the dataframe --------
df_nola = pd.DataFrame(restaurants_data)

print(f"Extration complete! Found {len(df_nola)} restaurants in the New Orleans")


Extration complete! Found 1480 restaurants in the New Orleans


In [ ]:
# -------- Missing Values Audit ---------

print("Calculating missing data percentage per column...")

# Calculate the percentage of Nans per columns
missing_data = df_nola.isnull().mean() * 100

# Filter only the columns with missing data and sort them in descending order
missing_data = missing_data[missing_data > 0].sort_values(ascending=False)

print("TOP 20 colunms with most missing data (%):")
display(missing_data.head(20))

# Check critical business variables
print("\n-> CRITICAL BUSINESS VARIABLES:")
if 'attr_RestaurantsPriceRange2' in df_nola.columns:
    missing_price = df_nola['attr_RestaurantsPriceRange2'].isnull().mean() * 100
    print(f"Price Range is missing in: {missing_price:.2f}% of restaurants")

if 'is_open' in df_nola.columns:
    missing_open = df_nola['is_open'].isnull().mean() * 100
    print(f"Business Status (Open/Closed) is missing in: {missing_open:.2f}% of restaurants")

Calculating missing data percentage per column...

-> CRITICAL BUSINESS VARIABLES:
Price Range is missing in: 12.36% of restaurants
Business Status (Open/Closed) is missing in: 0.00% of restaurants


In [ ]:
# -------- Data Preparation (CLEANING) --------- 

print("Cleaning critical variables...")

# Select all columns starting with 'attr_'
attr_columns = [col for col in df_nola.columns if col.startswith('attr_')]

# Fill NaNs in all attributes with the 'Unknown' category
df_nola[attr_columns] = df_nola[attr_columns].fillna('Unknown')

print("All missing categorical attributes filled with 'Unknown'.\n")

# Final safety check
if 'attr_RestaurantsPriceRange2' in df_nola.columns:
    missing_price_after = df_nola['attr_RestaurantsPriceRange2'].isnull().sum()
    print(f"Total missing in Price Range now: {missing_price_after}")

Cleaning critical variables...
All missing categorical attributes filled with 'Unknown'.

Total missing in Price Range now: 0


In [ ]:
# --------- Export Clean Data ---------

import os

# Ensure the data folder exists
os.makedirs('../data', exist_ok=True)

# Save the clean DataFrame to a CSV file
clean_file_path = '../data/nola_restaurants_clean.csv'
df_nola.to_csv(clean_file_path, index=False)

print(f"Success! The clean table was saved to: {clean_file_path}")

Success! The clean table was saved to: ../data/nola_restaurants_clean.csv
